In [ ]:
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np

# Pregunta 1

In [ ]:
# Histograma de tomates.png
tomates = cv.imread('Files/01_Prim/tomates.png', cv.IMREAD_GRAYSCALE)
plt.hist(tomates.ravel(),256,[0,256],color="black"); plt.show()

In [ ]:
# Binarización de tomates.png usando 3 umbrales
umbrales = [70, 120, 170]
for umbral in umbrales:
    _, temp = cv.threshold(tomates, umbral, 255, cv.THRESH_BINARY)
    plt.imshow(temp, cmap='gray')
    plt.axis('off')
    plt.show()

In [ ]:
# Mascara para detectar el tomate rojo utilizando el canal rojo
rojo = cv.imread('Files/01_Prim/tomates.png', cv.IMREAD_COLOR)[:,:,2]
_, mask = cv.threshold(rojo, 230, 255, cv.THRESH_BINARY)
plt.title("Tomates con máscara solo usando Rojo")
plt.imshow(mask, cmap='gray')
plt.axis('off')
plt.show()

# Mascara para detectar el tomate verde utilizando el canal rojo y verde
verde = cv.imread('Files/01_Prim/tomates.png', cv.IMREAD_COLOR)[:,:,1]
_, mask_verde = cv.threshold(verde, 160, 255, cv.THRESH_BINARY)
mask_verde = cv.bitwise_not(mask_verde)
_, mask_rojo = cv.threshold(rojo, 200, 255, cv.THRESH_BINARY)
mask_tomate_verde = cv.bitwise_and(mask_verde, mask_rojo)
plt.title("Tomates con máscara rojo y no verde")
plt.imshow(mask_tomate_verde, cmap='gray')
plt.axis('off')
plt.show()

# Pregunta 2

In [ ]:
image_path = 'Files/01_Prim/Marty.png'
img = cv.imread(image_path, cv.IMREAD_COLOR)
img = cv.cvtColor(img, cv.COLOR_BGR2RGB) 
plt.imshow(img)
plt.title("Imagen Original")
plt.axis('off')
plt.show()
#gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

#Usando la imagen "Marty.png", separe y binarice la imagen en los tres canales de color (rojo, verde, azul), aplicando un umbral de 50

marty_rojo = img[:, :, 2]
marty_verde = img[:, :, 1]
marty_azul = img[:, :, 0]


#aplicamos mascara de umbral
_,marty_rojo_bin = cv.threshold(marty_rojo, 50, 255, cv.THRESH_BINARY)
_,marty_verde_bin = cv.threshold(marty_verde, 50, 255, cv.THRESH_BINARY)
_,marty_azul_bin = cv.threshold(marty_azul, 50, 255, cv.THRESH_BINARY)
not_verde_marty = cv.bitwise_not(marty_verde_bin)

#combinamos las mascaras
mask = cv.bitwise_or(marty_rojo_bin, marty_azul_bin)
mask = cv.bitwise_or(mask, not_verde_marty)


#Mejoramos la mascara
kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (5, 5))

mask = cv.morphologyEx(mask, cv.MORPH_OPEN, kernel)
mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, kernel)
#errocionemos un poco mas la mascara para elmiminar las lineas de fondo
mask = cv.erode(mask, kernel, iterations=1)



plt.imshow(mask, cmap='gray')
plt.title("Máscara")
plt.axis('off')  # Quita ejes
plt.show()


In [ ]:
#Segmentamos a marty
segmentacion = cv.bitwise_and(img, img, mask=mask)
plt.imshow(segmentacion, cmap='gray')
plt.title("Marty recuperado")
plt.axis('off')  # Quita ejes
plt.show()

In [ ]:
not_mask = cv.bitwise_not(mask)
path_Paseo = "Files/01_Prim/PaseoBulnes.JPG"

paseobulnes = cv.imread(path_Paseo, cv.IMREAD_COLOR)
paseobulnes = cv.cvtColor(paseobulnes, cv.COLOR_BGR2RGB) 

paseobulnes = cv.bitwise_and(paseobulnes, paseobulnes, mask=not_mask)
plt.imshow(paseobulnes, cmap='gray')
plt.title("Máscara en Paseo Bulnes")
plt.axis('off')  # Quita ejes
plt.show()

In [ ]:
#pegamos a nuestro gallina
final = cv.add(segmentacion, paseobulnes)
plt.imshow(final, cmap='gray')
plt.title("Marty en Paseo Bulnes")
plt.axis('off')  # Quita ejes
plt.show()

# Pregunta 3

In [ ]:

img_path = "Files/01_Prim"
old = cv.imread(f'{img_path}/old.png')
pool = cv.imread(f'{img_path}/pool.png')
luna = cv.imread(f'{img_path}/moon.jpg')
luna = cv.cvtColor(luna, cv.COLOR_BGR2RGB)



def normalize_channel(channel, minimum=None, maximum=None):
    if minimum is None:
        min_val = channel.min()
    else:
        min_val = minimum
    if maximum is None:
        max_val = channel.max()
    else:
        max_val = maximum

    if max_val > min_val:  # evitar división por cero
        return (channel - min_val) / (max_val - min_val) * 255.0
    return np.zeros_like(channel)
def normalize_image(image, minimum=None, maximum=None):
    
    b, g, r = cv.split(image)
    b_norm = normalize_channel(b, minimum, maximum)
    g_norm = normalize_channel(g, minimum, maximum)
    r_norm = normalize_channel(r, minimum, maximum)

    # Recomponer y convertir a uint8
    return cv.merge([b_norm, g_norm, r_norm]).astype(np.uint8)
old_norm = normalize_image(old, 100, 255)
moon_norm = normalize_image(luna, 50, 255)
pool_norm = normalize_image(pool, 0, 105)
plt.imshow(old)
plt.title("Imagen vieja")
plt.show()

plt.hist(old.ravel(),256,[0,256])
plt.title('old Histogram')
plt.show()

plt.imshow(old_norm)
plt.title("Imagen vieja normalizada")
plt.show()
plt.hist(old_norm.ravel(),256,[0,256])
plt.title('Histograma imagen vieja normalizada')
plt.show()

plt.imshow(luna)
plt.title('Luna')
plt.show()

plt.hist(luna.ravel(),256,[0,256])
plt.title('Luna Histogram')
plt.show()

plt.imshow(moon_norm)
plt.title("luna normalizada")
plt.show()

plt.hist(moon_norm.ravel(),256,[0,256])
plt.title('Histograma imagen vieja normalizada')
plt.show()

plt.imshow(pool)
plt.title('Piscina')
plt.show()

plt.hist(pool.ravel(),256,[0,256])
plt.title('Pool Histogram')
plt.show()
plt.imshow(pool_norm)
plt.title('Piscina normalizada')
plt.show()
plt.hist(pool_norm.ravel(),256,[0,256])
plt.title('Histograma imagen vieja normalizada')
plt.show()

# Pregunta 4

In [ ]:
# Transformada de Fourier 2D y su visualización con y sin escala logarítmica
fence = cv.imread("Files/01_Prim/fence.png", cv.IMREAD_GRAYSCALE)
diagonal = cv.imread("Files/01_Prim/diagonal.jpg", cv.IMREAD_GRAYSCALE)


def fourier_transform(img_gray, title):
    # 1. Transformada de Fourier 2D
    f = np.fft.fft2(img_gray)
    fshift = np.fft.fftshift(f)
    magnitude = np.abs(fshift)

    # 2. Transformación logarítmica
    R = magnitude.max()
    c = 255 / np.log(1 + R)
    log_magnitude = c * np.log(1 + magnitude)
    fig, axs = plt.subplots(1, 3, figsize=(20,10))

    axs[0].imshow(img_gray, cmap='gray')
    axs[0].set_title(f'Imagen Original - {title}')
    axs[0].axis('off')

    axs[1].imshow(magnitude, cmap='gray')
    axs[1].set_title('FFT Magnitud (sin log)')
    axs[1].axis('off')

    axs[2].imshow(log_magnitude, cmap='gray')
    axs[2].set_title('FFT Magnitud (log)')
    axs[2].axis('off')

    plt.show()



fourier_transform(fence, "Fence")
fourier_transform(diagonal, "Diagonal")



# Pregunta 5

In [ ]:
# --- Función para FFT con log ---
def fft_log(img_gray):
    f = np.fft.fft2(img_gray)
    magnitude = np.abs(f)
    R = magnitude.max()
    c = 255 / np.log(1 + R)
    log_magnitude = c * np.log(1 + magnitude)
    return log_magnitude

def fft_shift(img_gray):
    f = np.fft.fft2(img_gray)
    fshift = np.fft.fftshift(f)
    magnitude = np.abs(fshift)
    log_magnitude = np.log(1 + magnitude)
    return log_magnitude

# --- Función para mostrar imágenes ---
def show_images(img_list, titles, suptitle=None, figsize=(15,5)):
    fig, axs = plt.subplots(1, len(img_list), figsize=figsize)
    for ax, img, title in zip(axs, img_list, titles):
        ax.imshow(img, cmap='gray')
        ax.set_title(title)
        ax.axis('off')
    if suptitle:
        fig.suptitle(suptitle, fontsize=20, y=1.05)  # título grande
    plt.tight_layout()
    plt.show()
# --- Cargar imagen ---
space = cv.imread("Files/01_Prim/space.png", cv.IMREAD_GRAYSCALE)

# --- Rotaciones ---
def rotate_image(img, angle):
    h, w = img.shape
    center = (w//2, h//2)
    M = cv.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv.warpAffine(img, M, (w, h), flags=cv.INTER_LINEAR, borderMode=cv.BORDER_REPLICATE)
    return rotated

space_45 = rotate_image(space, 45)
space_60 = rotate_image(space, 60)

# --- FFT logarítmica ---
fft_space = fft_log(space)
fft_45 = fft_log(space_45)
fft_60 = fft_log(space_60)

fft_shift_space = fft_shift(space)
fft_shift_45 = fft_shift(space_45)
fft_shift_60 = fft_shift(space_60)
# --- Visualización ---
show_images([space, space_45, space_60],
            ['Original', 'Rotada 45°', 'Rotada 60°'],
            suptitle="Imagen en el dominio espacial")

show_images([fft_space, fft_45, fft_60],
            ['Original', 'Rotada 45°', 'Rotada 60°'],
            suptitle="FFT Logarítmica sin shift")

show_images([fft_shift_space, fft_shift_45, fft_shift_60],
            ['Original', 'Rotada 45°', 'Rotada 60°'],
            suptitle="FFT Logarítmica con shift")


In [ ]:
# --- Función para FFT con log ---
def fft_log(img_gray):
    f = np.fft.fft2(img_gray)
    magnitude = np.abs(f)
    R = magnitude.max()
    c = 255 / np.log(1 + R)
    log_magnitude = c * np.log(1 + magnitude)
    return log_magnitude

def fft_shift(img_gray):
    f = np.fft.fft2(img_gray)
    fshift = np.fft.fftshift(f)
    magnitude = np.abs(fshift)
    log_magnitude = np.log(1 + magnitude)
    return log_magnitude

# --- Función para mostrar imágenes ---
def show_images(img_list, titles, suptitle=None, figsize=(15,5)):
    fig, axs = plt.subplots(1, len(img_list), figsize=figsize)
    for ax, img, title in zip(axs, img_list, titles):
        ax.imshow(img, cmap='gray')
        ax.set_title(title)
        ax.axis('off')
    if suptitle:
        fig.suptitle(suptitle, fontsize=20, y=1.05)  # título grande
    plt.tight_layout()
    plt.show()
# --- Cargar imagen ---
space = cv.imread("Files/01_Prim/space.png", cv.IMREAD_GRAYSCALE)

# --- Rotaciones ---
def rotate_image(img, angle):
    h, w = img.shape
    center = (w//2, h//2)
    M = cv.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv.warpAffine(img, M, (w, h), flags=cv.INTER_NEAREST, borderMode=cv.BORDER_REPLICATE)
    return rotated

def upscale_and_rotate(img, angle, scale_factor=10):
    # 1. Escalar (aumentar resolución)
    h, w = img.shape
    img_up = cv.resize(img, (w*scale_factor, h*scale_factor), interpolation=cv.INTER_NEAREST)
    
    # 2. Rotar sobre la versión ampliada
    h_up, w_up = img_up.shape
    center = (w_up // 2, h_up // 2)
    M = cv.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv.warpAffine(img_up, M, (w_up, h_up), flags=cv.INTER_NEAREST, borderMode=cv.BORDER_REPLICATE)
    
    return rotated


def rotate_image_sharp(img, angle):
    h, w = img.shape
    center = (w//2, h//2)
    M = cv.getRotationMatrix2D(center, angle, 1.0)

    cos = np.abs(M[0,0])
    sin = np.abs(M[0,1])
    new_w = int(h*sin + w*cos)
    new_h = int(h*cos + w*sin)
    
    M[0,2] += (new_w/2) - center[0]
    M[1,2] += (new_h/2) - center[1]
    
    rotated = cv.warpAffine(img, M, (new_w, new_h), flags=cv.INTER_LINEAR_EXACT, borderMode=cv.BORDER_CONSTANT, borderValue=0)
    return rotated

space = upscale_and_rotate(space, 0)
space_45 = upscale_and_rotate(space, 45)
space_60 = upscale_and_rotate(space, 60)

# --- FFT logarítmica ---
fft_space = fft_log(space)
fft_45 = fft_log(space_45)
fft_60 = fft_log(space_60)

fft_shift_space = fft_shift(space)
fft_shift_45 = fft_shift(space_45)
fft_shift_60 = fft_shift(space_60)
# --- Visualización ---
show_images([space, space_45, space_60],
            ['Original', 'Rotada 45°', 'Rotada 60°'],
            suptitle="Imagen en el dominio espacial")

show_images([fft_space, fft_45, fft_60],
            ['Original', 'Rotada 45°', 'Rotada 60°'],
            suptitle="FFT Logarítmica sin shift")

show_images([fft_shift_space, fft_shift_45, fft_shift_60],
            ['Original', 'Rotada 45°', 'Rotada 60°'],
            suptitle="FFT Logarítmica con shift")
